# AttentionVision: Exploratory Analysis & Architecture Inspection

This notebook inspects:
1. Dataset splits (`train`, `val`, `test`) and sample visualizations.
2. Stage-by-stage tensor shapes and parameter counts of `BaselineCNN` vs. `AttentionCNN`.
3. Intermediate spatial attention maps $M_s(F) \in (0, 1)^{B \times 1 \times H \times W}$ at Stage 2 ($16 \times 16$) and Stage 3 ($8 \times 8$).
4. Completed smoke-test and corruption sensitivity metrics.

In [1]:
import json
from pathlib import Path
import pandas as pd
import torch

from attentionvision.config import load_config
from attentionvision.data import build_dataloaders, inspect_dataset_bundle
from attentionvision.models import AttentionCNN, BaselineCNN
from attentionvision.utils import count_parameters, get_environment_info

print(json.dumps(get_environment_info(), indent=2))

In [2]:
# 1. Inspect Dataset Split Integrity
cfg = load_config('../configs/smoke.yaml')
bundle = build_dataloaders(cfg.data, seed=cfg.seed)
print(json.dumps(inspect_dataset_bundle(bundle), indent=2))

In [3]:
# 2. Compare BaselineCNN and AttentionCNN Parameter Counts & Shapes
base_model = BaselineCNN()
attn_model = AttentionCNN(attention_kernel_size=7, attention_residual=True, attention_stages=(2, 3))

x = torch.randn(2, 3, 32, 32)
base_logits = base_model(x)
attn_logits, attn_maps = attn_model(x, return_attention=True)

print(f'BaselineCNN parameters : {count_parameters(base_model):,}')
print(f'AttentionCNN parameters: {count_parameters(attn_model):,}')
print(f'Parameter overhead     : +{count_parameters(attn_model) - count_parameters(base_model)} params')
print(f'Output logits shape    : {tuple(attn_logits.shape)}')
for stage_name, amap in attn_maps.items():
    print(f'{stage_name} attention map shape: {tuple(amap.shape)}, range=[{amap.min().item():.4f}, {amap.max().item():.4f}]')

In [4]:
# 3. Load Completed Smoke-Test Comparison and Robustness Tables
summary_csv = Path('../results/smoke/comparison_summary.csv')
if summary_csv.exists():
    df_summary = pd.read_csv(summary_csv)
    display(df_summary)

rob_csv = Path('../results/smoke/robustness/robustness_results.csv')
if rob_csv.exists():
    df_rob = pd.read_csv(rob_csv)
    display(df_rob)